#  문서 군집화 소개와 실습 (Opinion Review 데이터 세트)
문서 군집화: 비슷한 텍스트 구성의 문서를 군집화（Clustering）하는 것.

텍스트 분류 기반의 문서 분류는 사전에 결정 카테고리 값을 가진 학습 데이터 세트가 필요한 데 반해, 문서 군집화는 학습 데이터 세트가 필요없는 비지도학습 기반으로 동작.

### **Opinion Review 데이터 세트를 이용한 문서 군집화 수행하기**

In [2]:
import pandas as pd
import glob, os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)

# 디렉터리 설정
path = r'/content/drive/MyDrive/ESAA/OB/topic'
# path로 지정한 디렉터리 밑에 있는 모든 .data 파일들의 파일명을 리스트로 취합
all_files = glob.glob(os.path.join(path, '*.data'))
filename_list = []
opinion_text = []

# 개별 파일들의 파일명은 filename_list 리스트로 취합.
# 개별 파일들의 파일 내용은 DataFrame 로딩 후 다시 string으로 변환하여 opinion_text 리스트로 취합
for file_ in all_files:
  # 개별 파일들 읽어서 DataFrame으로 생성
  df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')

  # 절대경로로 주어진 파일명을 가공. Linux에서 수행 시에는 아래 \\를 /변경.
  # 맨 마지막 .data 확장자도 제거
  filename_ = file_.split('\\')[-1]
  filename = filename_.split('.')[0]

  # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가..
  filename_list.append(filename)
  opinion_text.append(df.to_string())

# 파일명 리스트와 파일 내용 리스트를 DataFrame으로 생성.
document_df = pd.DataFrame({'filename': filename_list, 'opinion_text': opinion_text})
document_df.head()

,filename,opinion_text
0,/content/drive/MyDrive/ESAA/OB/topic/display_garmin_nuvi_255W_gps,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...
1,/content/drive/MyDrive/ESAA/OB/topic/battery-life_ipod_nano_8gb,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ..."
2,/content/drive/MyDrive/ESAA/OB/topic/satellite_garmin_nuvi_255W_gps,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ..."
3,/content/drive/MyDrive/ESAA/OB/topic/seats_honda_accord_2008,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...
4,/content/drive/MyDrive/ESAA/OB/topic/video_ipod_nano_8gb,"Another thing to consider was that I paid $50 less for the 750 and it came with the FM transmitter cable and a USB cord to connect it to your computer for updates and downloads .\n0 update and reroute much _more_ quickly than my other GPS .\n1 UPDATE ON THIS , It finally turned out that to see the elevation contours at lowe..."


 TF—IDF 형태로 피처 벡터화

In [7]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

def LemNormalize(text):
  lemmatizer = WordNetLemmatizer()

  return [lemmatizer.lemmatize(token) for token in word_tokenize(text)]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english', \
                             ngram_range=(1,2), min_df=0.05, max_df=0.85 )

#opinion_text 칼럼값으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

 K-평균을 적용.

In [10]:
from sklearn.cluster import KMeans

# 5개 집합으로 군집화 수행. 예제를 위해 동일한 클러스터링 결과 도출용 random_state=0
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_centers = km_cluster.cluster_centers_

In [11]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,/content/drive/MyDrive/ESAA/OB/topic/display_garmin_nuvi_255W_gps,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...,2
1,/content/drive/MyDrive/ESAA/OB/topic/battery-life_ipod_nano_8gb,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",3
2,/content/drive/MyDrive/ESAA/OB/topic/satellite_garmin_nuvi_255W_gps,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",0
3,/content/drive/MyDrive/ESAA/OB/topic/seats_honda_accord_2008,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...,2
4,/content/drive/MyDrive/ESAA/OB/topic/video_ipod_nano_8gb,"Another thing to consider was that I paid $50 less for the 750 and it came with the FM transmitter cable and a USB cord to connect it to your computer for updates and downloads .\n0 update and reroute much _more_ quickly than my other GPS .\n1 UPDATE ON THIS , It finally turned out that to see the elevation contours at lowe...",2


판다스 DataFrame의 sort_values（by=정렬칼럼명）를 수행하면 인자로 입력된 ‘정렬칼럼명’으로 데이터를 정렬할 수 있음.

In [15]:
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
31,/content/drive/MyDrive/ESAA/OB/topic/battery-life_amazon_kindle,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",0
5,/content/drive/MyDrive/ESAA/OB/topic/food_swissotel_chicago,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,0
24,/content/drive/MyDrive/ESAA/OB/topic/free_bestwestern_hotel_sfo,The food for our event was delicious .\n0 ...,0
34,/content/drive/MyDrive/ESAA/OB/topic/gas_mileage_toyota_camry_2007,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,0
10,/content/drive/MyDrive/ESAA/OB/topic/location_holiday_inn_london,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",0
6,/content/drive/MyDrive/ESAA/OB/topic/mileage_honda_accord_2008,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",0
32,/content/drive/MyDrive/ESAA/OB/topic/performance_honda_accord_2008,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,0
21,/content/drive/MyDrive/ESAA/OB/topic/quality_toyota_camry_2007,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
11,/content/drive/MyDrive/ESAA/OB/topic/rooms_bestwestern_hotel_sfo,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",0
47,/content/drive/MyDrive/ESAA/OB/topic/rooms_swissotel_chicago,"Great Location , Nice Rooms , H...",0


Cluster #0은 호텔에 대한 리뷰로 군집화돼 있음.

In [17]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
28,/content/drive/MyDrive/ESAA/OB/topic/interior_toyota_camry_2007,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,1
7,/content/drive/MyDrive/ESAA/OB/topic/keyboard_netbook_1005ha,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",1


Cluster #1은 킨들, 아이팟, 넷북 등의 포터블 전자기기 및 주요 구성요소（배터리, 키보드등）에 대한 리뷰로 군집화돼 있음.

In [18]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
16,/content/drive/MyDrive/ESAA/OB/topic/bathroom_bestwestern_hotel_sfo,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi...",2
40,/content/drive/MyDrive/ESAA/OB/topic/comfort_honda_accord_2008,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",2
0,/content/drive/MyDrive/ESAA/OB/topic/display_garmin_nuvi_255W_gps,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...,2
35,/content/drive/MyDrive/ESAA/OB/topic/eyesight-issues_amazon_kindle,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",2
37,/content/drive/MyDrive/ESAA/OB/topic/features_windows7,"It feels as easy to read as the K1 but doesn't seem any crisper to my eyes .\n0 the white is really GREY, and to avoid considerable eye, strain I had to refresh pages every other page .\n1 The dream has always been a portable electronic device that could hold a ton of reading material, automate subscriptions and fa...",2
20,/content/drive/MyDrive/ESAA/OB/topic/fonts_amazon_kindle,"I had to uninstall anti, virus and selected other programs, some of which did not have listings in the Programs and Features Control Panel section .\n0 This review briefly touches upon some of the key features and enhancements of Microsoft's latest OS .\n1 ...",2
17,/content/drive/MyDrive/ESAA/OB/topic/food_holiday_inn_london,"Being able to change the font sizes is awesome !\n0 For whatever reason, Amazon decided to make the Font on the Home Screen ...",2
33,/content/drive/MyDrive/ESAA/OB/topic/location_bestwestern_hotel_sfo,", I think the new keyboard rivals the great hp mini keyboards .\n0 Since the battery life difference is minimum, the only reason to upgrade would be to get the better keyboard .\n1 The keyboard is now as good as t...",2
9,/content/drive/MyDrive/ESAA/OB/topic/parking_bestwestern_hotel_sfo,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",2
23,/content/drive/MyDrive/ESAA/OB/topic/price_amazon_kindle,"The Eee Super Hybrid Engine utility lets users overclock or underclock their Eee PC's to boost performance or provide better battery life depending on their immediate requirements .\n0 In Super Performance mode CPU, Z shows the bus speed to increase up to 169 .\n1 One...",2


Cluster #2는 Cluster #1 과 비슷하게 킨들등이 군집에 포함돼 있지만, 주로 차량용내비게이션으로 군집이 구성돼 있음.

In [19]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
1,/content/drive/MyDrive/ESAA/OB/topic/battery-life_ipod_nano_8gb,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",3
49,/content/drive/MyDrive/ESAA/OB/topic/battery-life_netbook_1005ha,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,3
14,/content/drive/MyDrive/ESAA/OB/topic/buttons_amazon_kindle,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",3
22,/content/drive/MyDrive/ESAA/OB/topic/screen_netbook_1005ha,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ...",3
38,/content/drive/MyDrive/ESAA/OB/topic/speed_garmin_nuvi_255W_gps,headphone jack i got a clear case for it and it i got a clear case for it and it like prvents me from being able to put the jack all the way in so the sound can b messsed up or i can get it in there and its playing well them go to move or something and it slides out .\n0 Picture and sound quality are excellent for this typ of devic .\n1 ...,3
48,/content/drive/MyDrive/ESAA/OB/topic/voice_garmin_nuvi_255W_gps,"I bought the 8, gig Ipod Nano that has the built, in video camera .\n0 Itunes has an on, line store, where you may purchase and download music and videos which will install onto the ipod .\n1 ...",3


Cluster #3은 킨들(kindle) 리뷰가 한 개 섞여 있는 것이 살짝 아쉽지만, Cluster #0과 같이 대부분 호텔에 대한 리뷰로 군집화돼 있음.

In [20]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label
29,/content/drive/MyDrive/ESAA/OB/topic/comfort_toyota_camry_2007,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",4
44,/content/drive/MyDrive/ESAA/OB/topic/directions_garmin_nuvi_255W_gps,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",4
12,/content/drive/MyDrive/ESAA/OB/topic/interior_honda_accord_2008,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,4
26,/content/drive/MyDrive/ESAA/OB/topic/navigation_amazon_kindle,"It's quiet, get good gas mileage and looks clean inside and out .\n0 The mileage is great, and I've had to get used to stopping less for gas .\n1 Thought gas ...",4
36,/content/drive/MyDrive/ESAA/OB/topic/performance_netbook_1005ha,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser .\n0 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway .\n1 Overall performance is good but comfort level is poor .\n2 ...",4
18,/content/drive/MyDrive/ESAA/OB/topic/room_holiday_inn_london,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,4
45,/content/drive/MyDrive/ESAA/OB/topic/service_bestwestern_hotel_sfo,"Front seats are very uncomfortable .\n0 No memory seats, no trip computer, can only display outside temp with trip odometer .\n1 ...",4
19,/content/drive/MyDrive/ESAA/OB/topic/updates_garmin_nuvi_255W_gps,"After slowing down, transmission has to be kicked to speed up .\n0 ...",4


Cluster #4는 토요타(Toyota)와 혼다(Honda) 등의 자동차에 대한 리뷰로 잘 군집화돼 있음.

중심 개수를 5개에서 3개 그룹으로 군집화.

In [21]:
from sklearn.cluster import KMeans

# 3개의 집합으로 군집화
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

# 소속 클러스터를 cluster_label 컬럼으로 할당하고 cluster_label 값으로 정렬
document_df['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
2,/content/drive/MyDrive/ESAA/OB/topic/satellite_garmin_nuvi_255W_gps,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",0
5,/content/drive/MyDrive/ESAA/OB/topic/food_swissotel_chicago,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,0
6,/content/drive/MyDrive/ESAA/OB/topic/mileage_honda_accord_2008,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",0
11,/content/drive/MyDrive/ESAA/OB/topic/rooms_bestwestern_hotel_sfo,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",0
13,/content/drive/MyDrive/ESAA/OB/topic/service_swissotel_hotel_chicago,"not customer, oriented hotelvery low service levelboor reception\n0 The room was quiet, clean, the bed and pillows were comfortable, and the serv...",0
15,/content/drive/MyDrive/ESAA/OB/topic/staff_swissotel_chicago,Staff are friendl...,0
10,/content/drive/MyDrive/ESAA/OB/topic/location_holiday_inn_london,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",0
21,/content/drive/MyDrive/ESAA/OB/topic/quality_toyota_camry_2007,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
24,/content/drive/MyDrive/ESAA/OB/topic/free_bestwestern_hotel_sfo,The food for our event was delicious .\n0 ...,0
27,/content/drive/MyDrive/ESAA/OB/topic/size_asus_netbook_1005ha,Mediocre room and service for a very extravagant price .\n0 ...,0


### **군집별 핵심 단어 추출하기**
각 군집(Cluster)에 속한 문서는 핵심 단어를 주축으로 군집화돼 있을 것.
* KMeans 객체는 각 군집을 구성하는 단어 피처가 군집의 중심(Centroid)을 기준으로 얼마나 가깝게 위치해 있는지 clusters_centers_라는 속성으로 제공.
* clusters_centers_는 배열 값으로 제공되며, 행은 개별 군집을, 열은 개별 피처를 의미.
* 각 배열 내의 값은 개별 군집 내의 상대 위치를 숫자 값으로 표현한 일종의 좌표 값.
* cluster_centers_ 속성은 넘파이의 ndarray.
* ndarray의 argsort( )[:,::—1]를 이용하면 cluster_centers 배열 내 값이 큰 순으로 정렬된 위치 인덱스 값을 반환.(큰 값으로 정렬한 값을 반환하는 게 아니라 큰 값을 가진 배열 내 위치 인덱스 값을 반환하는 것)


In [22]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape ：', cluster_centers.shape)
print(cluster_centers)

cluster_centers shape ： (3, 6069)
[[0.         0.00041084 0.00090681 ... 0.00175958 0.00138842 0.00138842]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.00169252 0.00125009 0.         ... 0.         0.         0.        ]]


In [25]:
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 반환함.
def get_cluster_details(cluster_model, cluster_data, feature_names, cluster_num, top_n_features=10):
  cluster_details = {}

  # cluster_centers array의 값이 큰 순으로 정렬된 인덱스 값을 반환
  # 군집 중심점별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함.
  centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,::-1]

  # 개별 군집별로 반복하면서 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명 입력
  for cluster_num in range(cluster_num):
    # 개별 군집별 정보를 담을 데이터 초기화.
    cluster_details[cluster_num]= {}
    cluster_details[cluster_num]['cluster'] = cluster_num

    # cluster_centers_.argsort()[:,::-1]로 구한 인덱스를 이용해 top n 피처 단어를 구함.
    top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
    top_features = [feature_names[ind] for ind in top_feature_indexes]

    # top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값 구함.
    top_feature_values = cluster_model.cluster_centers_[cluster_num,top_feature_indexes].tolist()

    # cluster_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심위치 상댓값, 해당 파일명 입력
    cluster_details[cluster_num]['top_features'] = top_features
    cluster_details[cluster_num]['top_features_value'] = top_feature_values
    filenames = cluster_data[cluster_data['cluster_label'] == cluster_num]['filename']
    filenames = filenames.values.tolist()

    cluster_details[cluster_num]['filenames'] = filenames

  return cluster_details

In [34]:
def print_cluster_details(cluster_details):
  for cluster_num, cluster_detail in cluster_details.items():
    print('###### Cluster {0}'.format(cluster_num))
    print('Top features:', cluster_detail['top_features'])
    print('Reviews 파일명:', cluster_detail['filenames'][:7])
    print('==============================================')

In [35]:
feature_names = tfidf_vect.get_feature_names_out()

cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df, feature_names=feature_names, cluster_num=3, top_n_features=10)

print_cluster_details(cluster_details)

###### Cluster 0
Top features: ['room', 'hotel', 'service', 'staff', 'food', 'location', 'bathroom', 'clean', 'price', 'parking']
Reviews 파일명: ['/content/drive/MyDrive/ESAA/OB/topic/satellite_garmin_nuvi_255W_gps', '/content/drive/MyDrive/ESAA/OB/topic/food_swissotel_chicago', '/content/drive/MyDrive/ESAA/OB/topic/mileage_honda_accord_2008', '/content/drive/MyDrive/ESAA/OB/topic/location_holiday_inn_london', '/content/drive/MyDrive/ESAA/OB/topic/rooms_bestwestern_hotel_sfo', '/content/drive/MyDrive/ESAA/OB/topic/service_swissotel_hotel_chicago', '/content/drive/MyDrive/ESAA/OB/topic/staff_swissotel_chicago']
###### Cluster 1
Top features: ['interior', 'seat', 'mileage', 'comfortable', 'gas', 'gas mileage', 'transmission', 'car', 'performance', 'quality']
Reviews 파일명: ['/content/drive/MyDrive/ESAA/OB/topic/keyboard_netbook_1005ha', '/content/drive/MyDrive/ESAA/OB/topic/interior_honda_accord_2008', '/content/drive/MyDrive/ESAA/OB/topic/room_holiday_inn_london', '/content/drive/MyDrive/ES

포터블 전자제품 리뷰 군집인 Cluster #0에서는 screen, battery, battery life 등과 같은 화면과 배터리 수명 등이 핵심 단어로 군집화됨.

자동차 리뷰 군집인Cluster #1에서는‘interior’, ‘seat’, ‘mileage’, ‘comfortable’ 등과 같은 실내 인테리어, 좌석, 연료 효율 등이 핵심 단어로 군집화됨.

호텔 리뷰 군집인 Cluster #2에서는 room, hotel, service, staff 등 같은
방과 서비스 등이 핵심 단어로 군집화됨.